# Clasificador modal jerárquico híbrido — producción

Modelo oficial: N1 Gradient Boosting, N2 Random Forest y N3 Extra Trees. Dataset ampliado de 114 viajes y 445 escenarios; Raw/L1/L2/L3 se agrupan siempre por viaje físico.

In [9]:
import sys, json, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingClassifier, ExtraTreesClassifier, RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import balanced_accuracy_score, f1_score, classification_report, confusion_matrix
ROOT = Path.cwd()
while not (ROOT / 'pipeline_v3').exists(): ROOT = ROOT.parent
sys.path[:0] = [str(ROOT), str(ROOT/'pipeline_v3/calibration_and_diagnostics/modes_matrices_finetuning/random_forest_calibration')]
from pipeline_v3.src import config
from pipeline_v3.src.random_forest_contract import N1_FEATURES, N2_FEATURES, N3_FEATURES, HYBRID_HYPERPARAMETERS, BUS_PROBABILITY_THRESHOLD
from pipeline_v3.src.modal_classification import create_modal_evaluator
from experimentar_n1_caminar import make_features, M2I, MODES, WALK


ImportError: cannot import name 'N1_FEATURES' from 'pipeline_v3.src.random_forest_contract' (C:\Users\Eydan\OneDrive\Escritorio\ITESM\MAITEC Lab\Eventos Masivos\GPS_Emissions_Project_Pipeline-v2.0\pipeline_v3\src\random_forest_contract.py)

## Dataset y exclusión de etiquetas mixtas

In [ ]:
df = make_features('datos_entrenamiento_ml_expanded.pkl')
assert df.caid_trip.nunique() == 114 and len(df) == 445
clean = pd.read_csv(config.GPS_DIR/'Datos de MATLAB GPS Limpios.csv')
mixed = {f'{c}_{int(float(t))}' for (c,t),g in clean.groupby(['caid','num_trip']) if g.mode_of_transport.dropna().astype(str).str.strip().str.lower().nunique() > 1}
assert set(df.caid_trip).isdisjoint(mixed)
display(df.groupby('label').caid_trip.nunique().rename('viajes'))
display(df.label.value_counts().rename('escenarios'))


## Contratos ordenados: N1=16, N2=52, N3=25

In [ ]:
print('N1', len(N1_FEATURES), list(N1_FEATURES))
print('N2', len(N2_FEATURES), list(N2_FEATURES))
print('N3', len(N3_FEATURES), list(N3_FEATURES))
print('Umbral Bus', BUS_PROBABILITY_THRESHOLD)


## Validación agrupada de la cascada oficial

In [ ]:
X = df[list(N2_FEATURES)]; y = df.label.map(M2I).astype(int); groups = df.caid_trip
oof = np.zeros(len(df), dtype=int); folds = np.zeros(len(df), dtype=int)
cv = StratifiedGroupKFold(5, shuffle=True, random_state=42)
for fold, (tr, te) in enumerate(cv.split(X, y, groups)):
    assert set(groups.iloc[tr]).isdisjoint(set(groups.iloc[te]))
    n1 = GradientBoostingClassifier(**HYBRID_HYPERPARAMETERS['n1']).fit(X.iloc[tr][list(N1_FEATURES)], (y.iloc[tr] != WALK).astype(int))
    motor = y.iloc[tr] != WALK
    n2 = RandomForestClassifier(**HYBRID_HYPERPARAMETERS['n2']).fit(X.iloc[tr][motor][list(N2_FEATURES)], (y.iloc[tr][motor] == 2).astype(int))
    road = y.iloc[tr].isin([0, 1])
    n3 = ExtraTreesClassifier(**HYBRID_HYPERPARAMETERS['n3']).fit(X.iloc[tr][road][list(N3_FEATURES)], (y.iloc[tr][road] == 1).astype(int))
    p1 = n1.predict(X.iloc[te][list(N1_FEATURES)]); pred = np.full(len(te), WALK); mot = np.flatnonzero(p1 == 1)
    p2 = n2.predict(X.iloc[te].iloc[mot][list(N2_FEATURES)]); pred[mot[p2 == 1]] = 2; surface = mot[p2 == 0]
    pred[surface] = np.where(n3.predict_proba(X.iloc[te].iloc[surface][list(N3_FEATURES)])[:,1] >= BUS_PROBABILITY_THRESHOLD, 1, 0)
    oof[te] = pred; folds[te] = fold
print('Balanced Accuracy', balanced_accuracy_score(y, oof))
print('Macro F1', f1_score(y, oof, average='macro'))
print(classification_report(y, oof, target_names=MODES, digits=4))
cm = confusion_matrix(y, oof)
display(pd.DataFrame(cm, index=MODES, columns=MODES))
fig, ax = plt.subplots(figsize=(7, 6)); image = ax.imshow(cm, cmap='Blues')
for i in range(len(MODES)):
    for j in range(len(MODES)): ax.text(j, i, str(cm[i,j]), ha='center', va='center', color='white' if cm[i,j] > cm.max()/2 else 'black')
ax.set(xticks=range(4), yticks=range(4), xticklabels=MODES, yticklabels=MODES, xlabel='Predicción', ylabel='Clase real', title='Clasificador híbrido — predicciones OOF')
fig.colorbar(image, ax=ax); fig.tight_layout()
plot_dir = ROOT/'outputs/production_smoke_tests'; plot_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(plot_dir/'hybrid_oof_confusion_matrix.png', dpi=180, bbox_inches='tight'); plt.show()


In [ ]:
results = df[['deg']].copy(); results['y'] = y; results['pred'] = oof
by_deg = results.groupby('deg').apply(lambda g: pd.Series({'balanced_accuracy': balanced_accuracy_score(g.y,g.pred), 'macro_f1': f1_score(g.y,g.pred,average='macro')}), include_groups=False)
display(by_deg.loc[['Raw','L1','L2','L3']])


## Inferencia, guardrail y selección por configuración

In [ ]:
hybrid = create_modal_evaluator('hybrid')
rf_rollback = create_modal_evaluator('random_forest')
bayes = create_modal_evaluator('bayes')
print(type(hybrid).__name__, type(rf_rollback).__name__, type(bayes).__name__)
frame = pd.DataFrame({'caid':['DEMO']*20, 'trip':[1]*20, 'Speed [km/h]':np.linspace(12,30,20), 'local_timestamp':pd.date_range('2026-07-15',periods=20,freq='30s'), 'highway':['primary']*20, 'distance_m':[100.0]*20, 'near_bus_route':[0]*20, 'near_subway_line':[0]*20})
hybrid.raw_counts['DEMO_1'] = 20
hypotheses = {'Carro': frame}
print('Inferencia:', hybrid.select_final_mode(hypotheses)[:3])
short = {k:v.iloc[:10].copy() for k,v in hypotheses.items()}
print('Guardrail:', hybrid.select_final_mode(short)[0])  # Calidad insuficiente


## Historial

ML v4 usaba tres Random Forest y 66 viajes. Los experimentos de 49 variables y las seis variables Bus descartadas permanecen documentados en `archive/random_forest_experiments/`. El flujo principal actual es el clasificador modal jerárquico híbrido.